In [9]:
import os
import json
import joblib
import pandas as pd

from dotenv import load_dotenv
from groq import Groq
from jsonschema import validate


SET UP LLM CONNECTION

In [12]:
load_dotenv(override=True)

client = Groq(
    api_key=os.getenv("GROQ_API_KEY")
)

print("Groq Connected Successfully")

Groq Connected Successfully


In [13]:
model = joblib.load("../Part-3/best_model.pkl")

print(" Model Loaded Successfully")

 Model Loaded Successfully


In [14]:
df = pd.read_csv("../Part-1/cleaned_data.csv")

X = df.drop("price", axis=1)

cat_cols = X.select_dtypes(include=["object"]).columns

X = pd.get_dummies(
    X,
    columns=cat_cols,
    drop_first=True
)

X = X.fillna(X.mean(numeric_only=True))

print("Dataset Ready")

Dataset Ready


In [15]:
sample = X.iloc[[10,100,500]]

sample

,total_sqft,bath,balcony,area_type_Carpet Area,area_type_Plot Area,area_type_Super built-up Area,availability_14-Nov,availability_15-Aug,availability_15-Dec,availability_15-Jun,...,society_Xeitaa,society_YCnce R,society_YMhenLi,society_Yaenti,society_ZeodsWo,society_Zonce E,society_Zostaa,society_i1ncyRe,society_i1odsne,society_i1rtsCo
10,1800.0,2.0,2.0,False,False,True,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
100,2502.0,3.0,3.0,False,False,True,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
500,720.0,2.0,1.0,False,True,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [16]:
pred = model.predict(sample)

prob = model.predict_proba(sample)

print(pred)

print(prob)

[1 1 0]
[[0.44102294 0.55897706]
 [0.3678025  0.6321975 ]
 [0.53217688 0.46782312]]


In [17]:
schema = {

    "type":"object",

    "properties":{

        "prediction_label":{"type":"integer"},

        "confidence":{"type":"number"},

        "top_reason":{"type":"string"},

        "second_reason":{"type":"string"},

        "next_step":{"type":"string"}

    },

    "required":[

        "prediction_label",

        "confidence",

        "top_reason",

        "second_reason",

        "next_step"

    ]
}

In [18]:
def call_llm(system_prompt,user_prompt,temperature=0):

    if has_pii(user_prompt):

        return {
            "error":"PII Detected"
        }

    response = client.chat.completions.create(

        model="llama-3.3-70b-versatile",

        temperature=temperature,

        messages=[

            {
                "role":"system",
                "content":system_prompt
            },

            {
                "role":"user",
                "content":user_prompt
            }

        ]

    )

    return response.choices[0].message.content

PROMPT DESIGN

In [19]:
system_prompt = """
You are an AI assistant.

Return ONLY valid JSON.

Do not use markdown.

Do not explain.

Output must begin with { and end with }.
"""

user_prompt = f"""
Prediction:

{pred.tolist()}

Probability:

{prob.tolist()}

Explain prediction.

Return JSON with:

prediction_label

confidence

top_reason

second_reason

next_step
"""

In [22]:
def has_pii(text):
    return False

In [23]:
response = call_llm(

    system_prompt,

    user_prompt,

    temperature=0

)

print(response)

{
  "prediction_label": 1,
  "confidence": 0.6321975016708365,
  "top_reason": "High probability of class 1 in the second prediction",
  "second_reason": "Moderate probability of class 1 in the first prediction",
  "next_step": "Review and verify the prediction results"
}


JSON VALIDATON

In [24]:
result = json.loads(response)

validate(
    instance=result,
    schema=schema
)

print("✅ JSON Valid")
print(result)

✅ JSON Valid
{'prediction_label': 1, 'confidence': 0.6321975016708365, 'top_reason': 'High probability of class 1 in the second prediction', 'second_reason': 'Moderate probability of class 1 in the first prediction', 'next_step': 'Review and verify the prediction results'}


TEMPERATURE

In [25]:
response_temp0 = call_llm(
    system_prompt,
    user_prompt,
    temperature=0
)

response_temp07 = call_llm(
    system_prompt,
    user_prompt,
    temperature=0.7
)

print("===== Temperature = 0 =====")
print(response_temp0)

print("\n=========================\n")

print("===== Temperature = 0.7 =====")
print(response_temp07)

===== Temperature = 0 =====
{
  "prediction_label": 1,
  "confidence": 0.6321975016708365,
  "top_reason": "High probability of class 1 in the second prediction",
  "second_reason": "Moderate probability of class 1 in the first prediction",
  "next_step": "Review and verify the prediction results"
}


===== Temperature = 0.7 =====
{
  "prediction_label": 1,
  "confidence": 0.6321975016708365,
  "top_reason": "High probability of class 1 in the second prediction",
  "second_reason": "Moderate probability of class 1 in the first prediction",
  "next_step": "Review and verify the prediction results"
}


In [26]:
print("\n=========================\n")
print("END TO END DEMONSTRATION")
print("\n=========================\n")
sample = X.iloc[[10]]

prediction = model.predict(sample)[0]

probability = model.predict_proba(sample)[0]

system_prompt = """
You are an AI assistant.

Return ONLY valid JSON.

Do not use markdown.
"""

user_prompt = f"""
Prediction: {prediction}

Confidence: {max(probability)}

Explain the prediction.

Return JSON with:
prediction_label
confidence
top_reason
second_reason
next_step
"""

response = call_llm(system_prompt,user_prompt)

print("===== ML Prediction =====")
print(prediction)

print("\n===== Probability =====")
print(probability)

print("\n===== LLM Response =====")
print(response)

result = json.loads(response)

validate(
    instance=result,
    schema=schema
)

print("\n✅ JSON Validation Passed")

print(json.dumps(result,indent=4))



END TO END DEMONSTRATION


===== ML Prediction =====
1

===== Probability =====
[0.44102294 0.55897706]

===== LLM Response =====
{
  "prediction_label": 1,
  "confidence": 0.5589770580876143,
  "top_reason": "Insufficient data to provide a specific reason",
  "second_reason": "Unable to determine a secondary reason",
  "next_step": "Gather more information to improve prediction accuracy"
}

✅ JSON Validation Passed
{
    "prediction_label": 1,
    "confidence": 0.5589770580876143,
    "top_reason": "Insufficient data to provide a specific reason",
    "second_reason": "Unable to determine a secondary reason",
    "next_step": "Gather more information to improve prediction accuracy"
}


In [ ]:
#call LLM function
def call_llm(prompt):
        response = model.generate_content(prompt)
        return response.text

In [ ]:
test = call_llm("Explain Machine Learning in one sentence.")

print(test)

ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.5-pro
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.5-pro
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.5-pro
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.5-pro
Please retry in 39.053139071s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "gener

In [ ]:
import joblib
best_model = joblib.load("../Part-3/best_model.pkl")
print("Best model loaded successfully!")
print(type(best_model))

Best model loaded successfully!
<class 'sklearn.pipeline.Pipeline'>


In [ ]:
import pandas as pd

# Load cleaned dataset from Part 1
df = pd.read_csv("../Part-1/cleaned_data.csv")   # Change the filename if yours is different

print("Dataset Shape:", df.shape)
print(df.head())

Dataset Shape: (12790, 9)
              area_type   availability                  location       size  \
0  Super built-up  Area         19-Dec  Electronic City Phase II      2 BHK   
1            Plot  Area  Ready To Move          Chikka Tirupathi  4 Bedroom   
2        Built-up  Area  Ready To Move               Uttarahalli      3 BHK   
3  Super built-up  Area  Ready To Move        Lingadheeranahalli      3 BHK   
4  Super built-up  Area  Ready To Move                  Kothanur      2 BHK   

   society  total_sqft  bath  balcony   price  
0  Coomee       1056.0   2.0      1.0   39.07  
1  Theanmp      2600.0   5.0      3.0  120.00  
2      NaN      1440.0   2.0      3.0   62.00  
3  Soiewre      1521.0   3.0      1.0   95.00  
4      NaN      1200.0   2.0      1.0   51.00  


In [ ]:
X = df.drop("price", axis=1)

y_reg = df["price"]

median_price = y_reg.median()

y_clf = (y_reg >= median_price).astype(int)

cat_cols = X.select_dtypes(include=["object"]).columns

X = pd.get_dummies(
    X,
    columns=cat_cols,
    drop_first=True
)

X = X.fillna(
    X.mean(numeric_only=True)
)

X_train, X_test, y_clf_train, y_clf_test = train_test_split(
    X,
    y_clf,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(10232, 4107)
(2558, 4107)


In [ ]:
best_model = joblib.load(
    "../Part-3/best_model.pkl"
)

print(best_model)

Pipeline(steps=[('simpleimputer', SimpleImputer(strategy='median')),
                ('standardscaler', StandardScaler()),
                ('randomforestclassifier',
                 RandomForestClassifier(max_depth=10, n_estimators=50,
                                        random_state=42))])


In [ ]:
sample = X_test.iloc[[0]]

prediction = best_model.predict(sample)[0]

probability = best_model.predict_proba(sample)[0]

print(prediction)
print(probability)

1
[0.3819081 0.6180919]


In [ ]:
prompt = f"""
You are an AI assistant.

Prediction: {prediction}

Probability: {probability.tolist()}

Return ONLY valid JSON.

{{
    "prediction": "",
    "confidence": "",
    "explanation": "",
    "recommendation": ""
}}
"""

response = call_llm(prompt)

print(response)

ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.5-flash
Please retry in 528.65898ms. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerDayPerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-3.5-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 20
}
, retry_delay {
}
]

In [ ]:
import json

result = json.loads(response)

print(result)

TypeError: the JSON object must be str, bytes or bytearray, not GenerateContentResponse